<a href="https://colab.research.google.com/github/Aham0803/PySpark/blob/main/intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
! pip install pyspark

imported the required libraries

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg , col , lit, sum , count , desc,when,row_number
from pyspark.sql.window import Window

creating a session

In [2]:
spark = SparkSession.builder.appName("PySpark_Basics").getOrCreate()
print(f"spark session created successfullly with spark version :{spark.version}")

spark session created successfullly with spark version :4.0.3


create a random data

In [4]:
employee_data =[
    (101,"Gaurav yadav" , "IT" , 100000 ,"New Delhi"),
    (102,"Dev yadav" , "IT" , 80000 ,"Mumbai"),
    (103,"Gaurav kumar" , "Sales" , 70000 ,"Chennai"),
    (104,"Amit yadav" , "IT" , 100000 ,"New Delhi"),
    (105,"Divya yadav" , "HR" , 50000 ,"Banglore"),
    (106,"Ajay yadav" , "IT" , 90000 ,"Ahemdabad"),
    (106,"Gaurav Raj" , "IT" , 10000 ,"Pune"),
]
column = ["emp_id" , "emp_name" , "department" , "salary" ,"city"]
df = spark.createDataFrame(
    data = employee_data,
    schema = column
)

In [5]:
print("original schema\n")
df.show()
df.printSchema()

original schema

+------+------------+----------+------+---------+
|emp_id|    emp_name|department|salary|     city|
+------+------------+----------+------+---------+
|   101|Gaurav yadav|        IT|100000|New Delhi|
|   102|   Dev yadav|        IT| 80000|   Mumbai|
|   103|Gaurav kumar|     Sales| 70000|  Chennai|
|   104|  Amit yadav|        IT|100000|New Delhi|
|   105| Divya yadav|        HR| 50000| Banglore|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|
|   106|  Gaurav Raj|        IT| 10000|     Pune|
+------+------------+----------+------+---------+

root
 |-- emp_id: long (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)



Data Transformation

1 . Selecting and renaming columns

In [6]:
df_selected = df.select(col('emp_id') ,col('emp_name').alias ("full_name") , col('salary'))
df_selected.show(5)

+------+------------+------+
|emp_id|   full_name|salary|
+------+------------+------+
|   101|Gaurav yadav|100000|
|   102|   Dev yadav| 80000|
|   103|Gaurav kumar| 70000|
|   104|  Amit yadav|100000|
|   105| Divya yadav| 50000|
+------+------------+------+
only showing top 5 rows


2. filtering rows

In [7]:
df_filtered = df.filter((col('department') == "IT") & (col('salary') >80000))
df_filtered.show()

+------+------------+----------+------+---------+
|emp_id|    emp_name|department|salary|     city|
+------+------------+----------+------+---------+
|   101|Gaurav yadav|        IT|100000|New Delhi|
|   104|  Amit yadav|        IT|100000|New Delhi|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|
+------+------------+----------+------+---------+



3. adding and modifying columns

In [9]:
df_transformed = df.withColumn("bonus" , col ("salary") * 0.10).withColumn("salary_tier" , when(col("salary")>= 90000 , lit("High")).otherwise(lit("standard")))
df_transformed.show()

+------+------------+----------+------+---------+-------+-----------+
|emp_id|    emp_name|department|salary|     city|  bonus|salary_tier|
+------+------------+----------+------+---------+-------+-----------+
|   101|Gaurav yadav|        IT|100000|New Delhi|10000.0|       High|
|   102|   Dev yadav|        IT| 80000|   Mumbai| 8000.0|   standard|
|   103|Gaurav kumar|     Sales| 70000|  Chennai| 7000.0|   standard|
|   104|  Amit yadav|        IT|100000|New Delhi|10000.0|       High|
|   105| Divya yadav|        HR| 50000| Banglore| 5000.0|   standard|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad| 9000.0|       High|
|   106|  Gaurav Raj|        IT| 10000|     Pune| 1000.0|   standard|
+------+------------+----------+------+---------+-------+-----------+



In [10]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(salary_tier, CASE WHEN '`>=`('salary, 90000) THEN High ELSE standard END, None)]
+- Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#50]
   +- LogicalRDD [emp_id#0L, emp_name#1, department#2, salary#3L, city#4], false

== Analyzed Logical Plan ==
emp_id: bigint, emp_name: string, department: string, salary: bigint, city: string, bonus: double, salary_tier: string
Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, bonus#50, CASE WHEN (salary#3L >= cast(90000 as bigint)) THEN High ELSE standard END AS salary_tier#51]
+- Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#50]
   +- LogicalRDD [emp_id#0L, emp_name#1, department#2, salary#3L, city#4], false

== Optimized Logical Plan ==
Project [emp_id#0L, emp_name#1, department#2, salary#3L, city#4, (cast(salary#3L as double) * 0.1) AS bonus#50, CASE W

4. Aggregation and grouping

In [12]:
df_agg = df.groupby("department").agg(avg("salary").alias("avg_salary"),count("emp_id").alias("emp_count")).orderBy(desc("avg_salary"))
df_agg.show()

+----------+----------+---------+
|department|avg_salary|emp_count|
+----------+----------+---------+
|        IT|   76000.0|        5|
|     Sales|   70000.0|        1|
|        HR|   50000.0|        1|
+----------+----------+---------+



5. window function

In [14]:
window_spec = Window.partitionBy("department").orderBy(desc("salary"))
df_ranked = df.withColumn("salary_rank" , row_number().over(window_spec))
df_ranked.show()

+------+------------+----------+------+---------+-----------+
|emp_id|    emp_name|department|salary|     city|salary_rank|
+------+------------+----------+------+---------+-----------+
|   105| Divya yadav|        HR| 50000| Banglore|          1|
|   101|Gaurav yadav|        IT|100000|New Delhi|          1|
|   104|  Amit yadav|        IT|100000|New Delhi|          2|
|   106|  Ajay yadav|        IT| 90000|Ahemdabad|          3|
|   102|   Dev yadav|        IT| 80000|   Mumbai|          4|
|   106|  Gaurav Raj|        IT| 10000|     Pune|          5|
|   103|Gaurav kumar|     Sales| 70000|  Chennai|          1|
+------+------------+----------+------+---------+-----------+



stoppping the spark session

In [15]:
spark.stop()
print("\n Spark Session Stopped successfullly")


 Spark Session Stopped successfullly
